# 카테고리 나누기 : 분류과정(텍스트 확인) - **엑셀 추출용 코드 아님!!**



In [ ]:
import pandas as pd
import os
import re

# 1. 크림 관련 파일 로드
cream_files = [
    "16. 어성초 센텔라 레드 스팟 크림 30g.csv",
    "22. 어성초 70 수딩 크림 50ml.csv",
]

# 카테고리 구조 정의 (업데이트된 버전)
category_structure = {
    "피부 타입": {
        "건성/속건조": ["속건조", "속당김", "건조", "당김", "보습", "보습력", "촘촘", "겉돌지"],
        "지성/유분": ["유분", "기름", "번들", "오일리", "산뜻"],
        "민감성/자극": ["자극", "순함", "순해", "순한", "따갑", "안심", "예민", "민감"]
    },
    "효능": {
        "트러블/여드름": ["좁쌀", "여드름", "흉터", "뒤집어", "트러블", "뾰루지", "흔적", "화농", "모낭염"],
        "진정/붉은기": ["진정", "가라앉", "붉은기", "재생", "회복", "잠재워", "홍조", "열감"],
        "미백/톤업": ["미백", "밝아", "환해", "광채", "톤업", "잡티", "뽀얘"],
        "효과/변화": ["효과", "개선", "확실히", "꾸준히", "비포", "애프터", "변화"]
    },
    "사용감/제형": {
        "제형/흡수력": ["제형", "발림", "부드럽", "꾸덕", "묽은", "질감", "밀착", "수분", "촉촉", "쫀쫀", "물광", "수분감", "보들", "흡수", "스며", "흡수력"],
        "끈적임/무게감": ["끈적", "가벼운", "무거운"]
    },
    "사용 상황": {
        "화장/데일리": ["화장", "메이크업", "아침", "자기 전", "듬뿍", "데일리", "밀리", "팩토"],
        "계절/세안후": ["샤워", "세안", "세수", "직후", "마무리", "욕실", "첫단계", "봄", "여름", "가을", "겨울", "환절기", "더워", "추워", "에어컨", "히터"]
    },
    "기타 특성": {
        "향": ["향", "냄새", "무향", "향기"],
        "가치/편의": ["인생템", "정착", "재구매", "세일", "쟁여", "대용량", "가격", "펌프형", "편의성", "강추", "인정"]
    }
}

# 긍/부정 판단을 위한 간이 사전 (프로젝트용)
pos_words = ["좋아", "좋아용", "최고", "만족", "순해", "잘 ", "빠름", "도움", "굿", "추천"]
neg_words = ["별로", "아쉬워", "안 맞", "트러블 나", "따가", "느려", "비싸", "그닥"]

def judge_sentiment(text):
    for nw in neg_words:
        if nw in text: return "부정"
    for pw in pos_words:
        if pw in text: return "긍정"
    return "보통"

# 핵심 문장 추출 함수 (키워드가 포함된 문장만 가져옴)
def extract_snippet(text, keywords):
    sentences = re.split(r'[.!\?\n]', text)
    for s in sentences:
        for kw in keywords:
            if kw in s:
                return s.strip()[:100] # 최대 100자까지만 추출
    return text[:100]

final_rows = []

for f in cream_files:
    if os.path.exists(f):
        df = pd.read_csv(f)
        product_name = f.split('.')[1].strip() # 파일명에서 제품명 추출

        for _, row in df.iterrows():
            review_content = str(row['리뷰내용'])
            rating = row.get('별점', '-') # 별점 컬럼이 있는 경우

            # 모든 카테고리에 대해 매칭 검사
            for main_cat, sub_cats in category_structure.items():
                for sub_cat, keywords in sub_cats.items():
                    # 해당 소분류 키워드 중 하나라도 포함되어 있는지 확인
                    matched_keywords = [kw for kw in keywords if kw in review_content]

                    if matched_keywords:
                        snippet = extract_snippet(review_content, matched_keywords)
                        sentiment = judge_sentiment(review_content)

                        final_rows.append({
                            "제품명": product_name,
                            "대분류": main_cat,
                            "소분류(카테고리)": sub_cat,
                            "별점": rating,
                            "긍/부정": sentiment,
                            "매칭키워드": ", ".join(matched_keywords),
                            "리뷰일부(스니펫)": snippet,
                            "전체리뷰": review_content
                        })

# 결과 저장
result_df = pd.DataFrame(final_rows)
result_df.to_csv("analyzed_reviews_navigation.csv", index=False, encoding='utf-8-sig')

print(f"분석 완료! 총 {len(result_df)}개의 분류 데이터가 생성되었습니다.")

분석 완료! 총 1192개의 분류 데이터가 생성되었습니다.


In [ ]:
import pandas as pd
import os
import re
from collections import Counter

# 1. 파일 설정
product_files = {
    "크림류": [
        "16. 어성초 센텔라 레드 스팟 크림 30g.csv",
        "22. 어성초 70 수딩 크림 50ml.csv"
    ],
    "로션류": []
}

# 2. 카테고리 구조
category_structure = {
    "피부 타입": {
        "건성/속건조": ["속건조", "속당김", "건조", "당김", "보습", "보습력", "촘촘", "겉돌지"],
        "지성/유분": ["유분", "기름", "번들", "오일리", "산뜻"],
        "민감성/자극": ["자극", "순함", "순해", "순한", "따갑", "안심", "예민", "민감"]
    },
    "효능": {
        "트러블/여드름": ["좁쌀", "여드름", "흉터", "뒤집어", "트러블", "뾰루지", "흔적", "화농", "모낭염"],
        "진정/붉은기": ["진정", "가라앉", "붉은기", "재생", "회복", "잠재워", "홍조", "열감"],
        "미백/톤업": ["미백", "밝아", "환해", "광채", "톤업", "잡티", "뽀얘"],
        "효과/변화": ["효과", "개선", "확실히", "꾸준히", "비포", "애프터", "변화"]
    },
    "사용감/제형": {
        "제형/흡수력": ["제형", "발림", "부드럽", "꾸덕", "묽은", "질감", "밀착", "수분", "촉촉", "쫀쫀", "물광", "수분감", "보들", "흡수", "스며", "흡수력"],
        "끈적임/무게감": ["끈적", "가벼운", "무거운"]
    },
    "사용 상황": {
        "화장/데일리": ["화장", "메이크업", "아침", "자기 전", "듬뿍", "데일리", "밀리", "팩토"],
        "계절/세안후": ["샤워", "세안", "세수", "직후", "마무리", "욕실", "첫단계", "봄", "여름", "가을", "겨울", "환절기", "더워", "추워", "에어컨", "히터"]
    },
    "기타 특성": {
        "향": ["향", "냄새", "무향", "향기"],
        "가치/편의": ["인생템", "정착", "재구매", "세일", "쟁여", "대용량", "가격", "통", "펌프형", "편의성", "인정", "강추", "추천", "제발"]
    }
}

pos_words = ["좋아", "최고", "만족", "순해", "굿", "추천", "인정", "강추", "찰떡"]
neg_words = ["별로", "아쉬워", "안 맞", "트러블 나", "따가", "비싸", "밀려"]

def judge_sentiment(text):
    for nw in neg_words:
        if nw in text: return "부정"
    for pw in pos_words:
        if pw in text: return "긍정"
    return "보통"

def extract_snippet(text, all_matched_keywords):
    sentences = re.split(r'[.!\?\n]', text)
    for s in sentences:
        if any(kw in s for kw in all_matched_keywords):
            return s.strip()[:80]
    return text[:80]

# --- 분석 시작 ---
final_rows = []
type_keyword_counts = {p_type: Counter() for p_type in product_files.keys()}

for p_type, files in product_files.items():
    for f in files:
        if os.path.exists(f):
            df = pd.read_csv(f)
            product_name = f.split('.')[1].strip() if '.' in f else f

            for _, row in df.iterrows():
                review_content = str(row['리뷰내용'])
                rating = row.get('별점', '-')
                sentiment = judge_sentiment(review_content)

                # 대분류별로 매칭 정보를 담을 딕셔너리
                # { "효능": {"sub_cats": ["진정", "트러블"], "keywords": ["붉은기", "좁쌀"]} }
                main_cat_matches = {}

                for main_cat, sub_cats in category_structure.items():
                    for sub_cat, keywords in sub_cats.items():
                        matched_kws = [kw for kw in keywords if kw in review_content]

                        if matched_kws:
                            if main_cat not in main_cat_matches:
                                main_cat_matches[main_cat] = {"sub_cats": [], "keywords": []}

                            main_cat_matches[main_cat]["sub_cats"].append(sub_cat)
                            main_cat_matches[main_cat]["keywords"].extend(matched_kws)

                            for kw in matched_kws:
                                type_keyword_counts[p_type][kw] += 1

                # 대분류 개수가 곧 정보 밀도
                density = len(main_cat_matches)

                # 대분류별로 행 생성 (같은 대분류 내 소분류는 합침)
                for main_cat, data in main_cat_matches.items():
                    # 중복 제거 및 콤마 연결
                    combined_sub_cats = ", ".join(sorted(list(set(data["sub_cats"]))))
                    combined_keywords = ", ".join(list(set(data["keywords"])))
                    snippet = extract_snippet(review_content, data["keywords"])

                    final_rows.append({
                        "제품군": p_type,
                        "제품명": product_name,
                        "대분류": main_cat,
                        "소분류": combined_sub_cats, # 여기서 합쳐짐
                        "정보밀도_카운트": density,
                        "별점": rating,
                        "긍부정": sentiment,
                        "매칭키워드": combined_keywords,
                        "리뷰일부": snippet,
                        "전체리뷰": review_content
                    })

# 3. 출력 및 저장
print("\n" + "="*60)
print(" [제품 제형별 핵심 키워드 언급 순위 (Top 10)] ")
print("="*60)

for p_type, counts in type_keyword_counts.items():
    if counts:
        print(f"\n▶ {p_type} 분석 결과:")
        for i, (word, freq) in enumerate(counts.most_common(10), 1):
            print(f"  {i}위. {word} ({freq}회)")

result_df = pd.DataFrame(final_rows)
column_order = ["제품군", "제품명", "대분류", "소분류", "정보밀도_카운트", "별점", "긍부정", "매칭키워드", "리뷰일부", "전체리뷰"]
result_df = result_df[column_order]
result_df.to_csv("final_review_navigation_data.csv", index=False, encoding='utf-8-sig')

print("\n" + "-"*60)
print(f"✔ 분석 데이터 저장 완료: final_review_navigation_data.csv")
print(f"✔ 중복 제거 후 데이터 행 수: {len(result_df)}개")


 [제품 제형별 핵심 키워드 언급 순위 (Top 10)] 

▶ 크림류 분석 결과:
  1위. 효과 (214회)
  2위. 진정 (176회)
  3위. 여드름 (137회)
  4위. 트러블 (95회)
  5위. 자극 (66회)
  6위. 꾸준히 (63회)
  7위. 재구매 (51회)
  8위. 통 (45회)
  9위. 추천 (45회)
  10위. 가라앉 (40회)

------------------------------------------------------------
✔ 분석 데이터 저장 완료: final_review_navigation_data.csv
✔ 중복 제거 후 데이터 행 수: 926개


# 파이널

S : Serum (세럼)
T : Toner (토너)
L : Lotion / Milk (로션/밀크)
C : Cream (크림)
A : Ampoule (앰플)
F : Foam (폼)
CO : Cleansing Oil (클렌징 오일)
SC : Sun Care / Sun Cream (선케어)


---



제품 특징으로 뒷 알파벳 설정
S (세럼) + R (라이스) = SR (라이스 세럼)

T (토너) + P (복숭아) = TP (복숭아 토너)

L (로션) + L (데일리/어성초) = LL (어성초 데일리 로션)

F (폼) + C (클렌징) = FC (모공 클렌징 폼)

### 세럼

In [12]:
import pandas as pd
import os
import re

# ==========================================
# 1. 제품별 식별 코드 매핑 (세럼 라인업)
# ==========================================
product_code_map = {
    "reviews_cleaned_1.아누아 피디알엔 히알루론산 캡슐 100 세럼.csv": "SH",
    "reviews_cleaned_4. 아누아_TXA_나이아신_흔적_세럼_30ml_2입.csv": "ST",
    "reviews_cleaned_13. 피디알엔 히알루론산 캡슐 100 세럼 1ml_10ea.csv": "SH2",
    "reviews_cleaned_14. 7 라이스 세라마이드 하이드레이팅 베리어 세럼 50ml.csv": "SR",
    "reviews_cleaned_17. TXA 나이아신 흔적 세럼 30ml.csv": "ST2",
    "reviews_cleaned_20. 복숭아 나이아신아마이드 세럼 30mL.csv": "SP"
}

# ==========================================
# 2. 고도화된 카테고리 구조 (Taxonomy 2.2)
# ==========================================
category_structure = {
    "피부 고민 해결": {
        "트러블/모공": ["여드름", "좁쌀", "트러블", "뾰루지", "모공", "요철", "블랙헤드", "화이트헤드", "흉터", "흔적", "화농성", "올라옴"],
        "진정/장벽": ["진정", "가라앉", "붉은기", "홍조", "재생", "회복", "장벽", "손상", "보호", "예민", "열감", "쿨링", "편안"],
        "미백/광채": ["미백", "밝아", "환해", "광채", "물광", "속광", "톤업", "잡티", "기미", "투명", "칙칙", "결 개선", "생기"]
    },
    "수분/유분 밸런스": {
        "보습/속건조": ["속건조", "속당김", "당김", "건조", "보습", "촉촉", "수분감", "푸석", "겉돌지", "수분충전", "촘촘", "속보습", "수부지"],
        "유분/피지": ["유분", "기름", "개기름", "번들", "산뜻", "피지", "보송", "오일리", "유분기", "답답함", "기름기"]
    },
    "제형/사용감": {
        "흡수/발림성": ["흡수", "스며", "발림", "제형", "질감", "밀착", "부드럽", "실키", "가벼운", "무거운", "점성", "묽은", "쫀쫀"],
        "끈적임/밀림": ["끈적", "밀림", "밀려", "밀착력", "잔여감", "겉돎", "유분감"]
    },
    "피부 자극": {
        "순함/저자극": ["순함", "순해", "순한", "자극없", "안심", "민감", "무자극", "따갑지", "순하게", "부담없는"],
        "자극/부작용": ["자극", "따갑", "따끔", "가려", "화끈", "뒤집", "붉어짐", "화끈거림", "가렵"]
    },
    "구매 가치/편의": {
        "가성비/가격": ["가격", "가성비", "세일", "혜택", "비쌈", "저렴", "대용량", "구성", "증정", "가심비", "착한", "착하다", "혜자", "이득", "사악"],
        "추천/재구매": ["추천", "강추", "인생템", "정착", "재구매", "지인", "비추", "선물", "만족", "재구입", "공병", "애정템"],
        "상황/편의": ["화장", "메이크업", "화잘먹", "펌프", "냄새", "향", "데일리", "팩토", "닦토", "흡토", "분사"]
    }
}

# ==========================================
# 3. [통합] 감성 사전 및 문맥 차단 로직
# ==========================================
SUPER_POS = ["인생템", "강추", "최고", "정착", "성공", "재구매", "무조건", "대박", "완전 추천", "효과 확실", "공병템", "확실", "아주 좋"]
NORMAL_POS = ["좋", "만족", "순해", "인정", "찰떡", "촉촉", "진정", "개선", "효과", "괜찮", "편안", "매끈", "맑", "탄탄", "믿고", "추천", "착한", "착하다", "혜자", "이득", "쫀쫀", "잘먹"]
SUPER_NEG = ["비추", "돈아깝", "거름", "최악", "다신안", "환불", "반품", "쓰레기", "뒤집어짐", "실망", "속았", "사악"]
NORMAL_NEG = ["아쉬", "안맞", "따가", "비싸", "밀려", "자극", "가려", "좁쌀", "번들", "답답", "냄새", "부족", "별로", "그다지", "불편", "심하", "의심", "부담되는", "들뜸", "뜸"]
INTENSIFIERS = ["너무", "진짜", "완전", "엄청", "진심", "정말", "되게", "매우", "솔직히", "겁나", "아주"]

ABSOLUTE_POS_PHRASES = ["화잘먹", "공병", "정착", "인생", "득템", "잘 먹어", "잘 먹음"]
QUOTE_MARKERS = ["광고", "바이럴", "영상", "유튜브", "남들은", "후기", "기대", "없어졌다고"]
CONTRAST_MARKERS = ["했는데", "했으나", "그치만", "하지만"]

NEGATIVE_REVERSAL = ["없", "않", "적", "사라", "진정", "개선", "나아", "제로", "줄어", "잡혔", "안 나", "안 뜨", "들어갔"]
NEG_TARGETS = ["자극", "끈적", "트러블", "밀림", "따가", "붉은기", "번들", "건조", "당김", "뒤집", "들뜸", "뜸", "유분"]

def smart_split(text):
    text = str(text)
    initial_splits = re.split(r'[.!\?\n]+', text)
    refined = []
    for s in initial_splits:
        s = s.strip()
        if len(s) > 35: # 문장이 너무 길면 연결어미 기준 2차 분리
            sub = re.split(r'(?<=[고서데며])\s+', s)
            refined.extend(sub)
        else:
            refined.append(s)
    return [s.strip() for s in refined if len(s.strip()) > 2]

def calculate_refined_sentiment(sentence, matched_kws):
    score = 0
    # 0. 광고/인용구 및 대조군 분석 (베이스 코드의 강점 반영)
    is_quote = any(q in sentence for q in QUOTE_MARKERS)
    has_contrast = any(c in sentence for c in CONTRAST_MARKERS)

    # 1. 기본 사전 점수
    for wp in SUPER_POS: score += 3 if wp in sentence else 0
    for wp in NORMAL_POS: score += 1 if wp in sentence else 0
    for wn in SUPER_NEG: score -= 3 if wn in sentence else 0
    for wn in NORMAL_NEG: score -= 1 if wn in sentence else 0

    # 2. 절대 긍정 구절 가점
    if any(ap in sentence for ap in ABSOLUTE_POS_PHRASES):
        score += 2

    # 3. 강조어 처리
    if any(i in sentence for i in INTENSIFIERS):
        if score > 0: score += 1
        elif score < 0: score -= 1

    # 4. 부정 요소 반전 및 악화 로직 (고도화)
    for target in NEG_TARGETS:
        if target in sentence:
            after_target = sentence.split(target)[-1]

            # 개선/진정 시 (부정어 + 반전어)
            if not is_quote and any(rev in after_target for rev in NEGATIVE_REVERSAL):
                score = abs(score) + 2

            # 악화 시 (부정어 + 악화어)
            if any(neg in after_target for neg in ["올라", "나왔", "생겼", "심해", "안 좋아"]):
                score -= 3

    # 5. 광고/대조군 문장 보정 (베이스 코드 로직 결합)
    if is_quote and has_contrast:
        if any(n in sentence for n in NORMAL_NEG + SUPER_NEG + ["올라", "안 좋아", "나왔"]):
            score = -4

    # 6. 가격 가점/감점
    if "가격" in sentence or "가성비" in sentence:
        if any(p in sentence for p in ["착한", "착하다", "혜자", "부담없", "저렴", "이득"]):
            score = max(score, 2)
        if any(n in sentence for n in ["사악", "비싸", "부담되"]):
            score = min(score, -2)

    score = int(max(-5, min(5, score)))
    label = "긍정" if score > 0 else "부정" if score < 0 else "보통"
    return label, score

# ==========================================
# 4. 분석 실행 및 리포트 저장
# ==========================================
final_rows = []

for f, p_code in product_code_map.items():
    if not os.path.exists(f): continue
    df = pd.read_csv(f)
    p_name = f.split('.')[-2].strip() if '.' in f else f
    df['날짜'] = df['날짜'].astype(str)
    df = df.sort_values('날짜')
    date_id_counts = {}

    for _, row in df.iterrows():
        date_raw = row['날짜']
        date_clean = date_raw.replace('.', '')[2:8]
        date_id_counts[date_clean] = date_id_counts.get(date_clean, 0) + 1

        review_id = f"{p_code}_{date_clean}_{date_id_counts[date_clean]:03d}"
        raw_text = str(row.get('clean_text', row.get('리뷰내용', '')))
        sentences = smart_split(raw_text)

        review_sentences_data = []
        review_all_kws, review_major_cats = set(), set()

        for sent in sentences:
            sent_major, sent_minor, sent_kws = set(), set(), set()
            for major, submap in category_structure.items():
                for minor, kws in submap.items():
                    matched = [kw for kw in kws if kw in sent]
                    if matched:
                        sent_major.add(major); sent_minor.add(minor); sent_kws.update(matched)
                        review_major_cats.add(major); review_all_kws.update(matched)

            if sent_kws:
                label, score = calculate_refined_sentiment(sent, sent_kws)
                review_sentences_data.append({"문장": sent, "라벨": label, "점수": score, "대": sent_major, "소": sent_minor, "키": sent_kws})

        avg_score = round(sum(d["점수"] for d in review_sentences_data) / len(review_sentences_data), 1) if review_sentences_data else 0.0

        for res in review_sentences_data:
            final_rows.append({
                "리뷰ID": review_id,
                "제품군": "세럼",
                "제품명": p_name,
                "날짜": date_raw,
                "가격": row.get('price', row.get('가격', '-')),
                "스킨타입": row.get('스킨타입', '-'),
                "태그": row.get('태그', '-'),
                "대분류": ", ".join(sorted(res["대"])),
                "소분류": ", ".join(sorted(res["소"])),
                "정보밀도(대분류수)": len(review_major_cats),
                "별점": row.get('별점', '-'),
                "긍부정": res["라벨"],
                "감성점수": res["점수"],
                "매칭키워드": ", ".join(sorted(res["키"])),
                "추출문장": res["문장"],
                "clean_text": raw_text,
                "리뷰_평균점수": avg_score,
                "전체_매칭키워드": ", ".join(sorted(review_all_kws))
            })

pd.DataFrame(final_rows).to_csv("serum_analysis_report.csv", index=False, encoding='utf-8-sig')
print(f"✔ 분석 완료: serum_analysis_report.csv")

✔ 분석 완료: serum_analysis_report.csv


In [13]:
import pandas as pd
import numpy as np
import os

# 1. 원본 데이터 로드 (기존 문장 단위 분석 결과 파일)
base_file = "serum_analysis_report.csv"

if not os.path.exists(base_file):
    print(f"⚠ {base_file} 파일이 없습니다. 분석을 먼저 실행해주세요.")
else:
    df = pd.read_csv(base_file)

    # 2. 속성(스킨타입 + 태그) 통합 및 분리 함수 (정보없음 포함 로직)
    def get_attribute_df(dataframe):
        temp_df = dataframe.copy()

        def combine_attrs(row):
            s = str(row.get('스킨타입', '-')).strip()
            t = str(row.get('태그', '-')).strip()

            # 비어있거나 의미 없는 값 체크
            def is_empty(val):
                return val in ['-', 'nan', 'None', '', 'nan ']

            attrs = []
            if not is_empty(s):
                attrs.extend([item.strip() for item in s.split(",")])
            if not is_empty(t):
                attrs.extend([item.strip() for item in t.split(",")])

            # 둘 다 정보가 없는 경우에만 '정보없음' 할당
            if not attrs:
                return ["정보없음"]
            # 중복 제거 (스킨타입과 태그에 중복된 단어가 있을 경우 대비)
            return list(set(attrs))

        temp_df['Attribute'] = temp_df.apply(combine_attrs, axis=1)
        # 리스트로 묶인 속성들을 행별로 분리(explode)
        temp_df = temp_df.explode('Attribute')
        return temp_df

    attr_df = get_attribute_df(df)
    final_report_rows = []
    products = df['제품명'].unique()

    for product in products:
        p_df = df[df['제품명'] == product]
        p_attr_df = attr_df[attr_df['제품명'] == product]

        # --- PART A: 속성별 수치 요약 (정보없음 포함) ---
        # 가나다 순 정렬 (정보없음이 포함됨)
        attributes = sorted(p_attr_df['Attribute'].unique())
        for attr in attributes:
            group = p_attr_df[p_attr_df['Attribute'] == attr]
            user_count = group['리뷰ID'].nunique()
            avg_score = round(group['감성점수'].mean(), 2)

            # 소분류(Sub-category) 추출 및 집계
            subcat_df = group.copy()
            subcat_df['Subcat'] = subcat_df['소분류'].astype(str).str.split(", ")
            subcat_df = subcat_df.explode('Subcat')
            subcat_df = subcat_df[subcat_df['Subcat'] != 'nan']

            pos_subs = subcat_df[subcat_df['긍부정'] == '긍정']['Subcat'].value_counts().head(5)
            pos_str = "\n".join([f"{k} ({v})" for k, v in pos_subs.items()]) if not pos_subs.empty else "-"

            neg_subs = subcat_df[subcat_df['긍부정'] == '부정']['Subcat'].value_counts().head(5)
            neg_str = "\n".join([f"{k} ({v})" for k, v in neg_subs.items()]) if not neg_subs.empty else "-"

            final_report_rows.append({
                "제품명": product,
                "상세 항목 및 요약": attr,
                "사용자 수 / 정보밀도": f"{user_count}명",
                "긍정 소분류 (Top 5)": pos_str,
                "부정 소분류 (Top 5)": neg_str,
                "종합 만족도": "매우만족" if avg_score >= 1.5 else "만족" if avg_score >= 0.5 else "보통",
                "평균 점수": avg_score
            })

        # --- PART B: 제품 총평 랩업 (Wrap-up) ---
        # 속성별 유니크 유저 수 집계
        demo_counts = p_attr_df.groupby('Attribute')['리뷰ID'].nunique().sort_values(ascending=False).to_dict()
        demo_str = ", ".join([f"{k} {v}명" for k, v in demo_counts.items()])

        # 제품 전체에서 가장 많이 언급된 특징과 감성
        overall_subcat_df = p_df.copy()
        overall_subcat_df['Subcat'] = overall_subcat_df['소분류'].astype(str).str.split(", ")
        overall_subcat_df = overall_subcat_df.explode('Subcat')
        overall_subcat_df = overall_subcat_df[overall_subcat_df['Subcat'] != 'nan']

        if not overall_subcat_df.empty:
            top_sub = overall_subcat_df['Subcat'].value_counts().idxmax()
            top_sub_score = overall_subcat_df[overall_subcat_df['Subcat'] == top_sub]['감성점수'].mean()
            top_sub_sent = "긍정" if top_sub_score > 0 else "부정"
        else:
            top_sub = "정보없음"
            top_sub_sent = "보통"

        wrap_up_text = f"[{product} 랩업] 참여 유저: ({demo_str}). 이 제품에서 가장 많이 언급된 특징은 '{top_sub}'(으)로, 이에 대해 주로 {top_sub_sent}적인 반응을 보였습니다."

        final_report_rows.append({
            "제품명": product,
            "상세 항목 및 요약": wrap_up_text,
            "사용자 수 / 정보밀도": f"총 {p_df['리뷰ID'].nunique()}명",
            "긍정 소분류 (Top 5)": "★★ 제품 전체 요약 ★★",
            "부정 소분류 (Top 5)": "★★ 제품 전체 요약 ★★",
            "종합 만족도": "-",
            "평균 점수": round(p_df['감성점수'].mean(), 2)
        })

        # --- PART C: 정보 밀도 높은 대표 리뷰 TOP 3 ---
        # 중복 제거 후 정보밀도 순으로 정렬
        top_reviews = p_df.drop_duplicates(subset='리뷰ID').sort_values(by='정보밀도(대분류수)', ascending=False).head(3)

        for i, (_, r_row) in enumerate(top_reviews.iterrows()):
            final_report_rows.append({
                "제품명": product,
                "상세 항목 및 요약": f"[대표리뷰 {i+1}] {r_row['clean_text']}",
                "사용자 수 / 정보밀도": f"밀도: {r_row['정보밀도(대분류수)']}",
                # [수정] 작성자 정보 자리에 고유 리뷰ID를 표시합니다.
                "긍정 소분류 (Top 5)": f"리뷰ID: {r_row['리뷰ID']}",
                "부정 소분류 (Top 5)": f"태그: {r_row['태그']}",
                "종합 만족도": f"별점: {r_row['별점']}",
                "평균 점수": r_row['리뷰_평균점수']
            })

    # 최종 결과 저장 (UTF-8-SIG 인코딩으로 엑셀 한글 깨짐 방지)
    final_df = pd.DataFrame(final_report_rows)
    final_df.to_csv("serum_summary_report.csv", index=False, encoding='utf-8-sig')
    print("✔ 소비자 인사이트 요약 리포트 생성 완료: serum_summary_report.csv")

✔ 소비자 인사이트 요약 리포트 생성 완료: serum_summary_report.csv


### 로션/크림/밀크

In [14]:
import pandas as pd
import os
import re

# ==========================================
# 1. 제품별 식별 코드 매핑 (로션/밀크/크림 라인업)
# ==========================================
product_code_map = {
    "reviews_cleaned_15. 어성초 70 데일리 로션 200ml.csv": "LL",
    "reviews_cleaned_18. 라이스 70 인텐시브 모이스처라이징 밀크 150ml.csv": "LR",
    "reviews_cleaned_19. 복숭아 77 나이아신 컨디셔닝 밀크 150ml.csv": "LP",
    "reviews_cleaned_16. 어성초 센텔라 레드 스팟 크림 30g.csv": "CS",
    "reviews_cleaned_22. 어성초 70 수딩 크림 50ml.csv": "CL",
    "reviews_cleaned_5.아누아 피디알엔 히알루론산 100 수분크림 60ml 기획 2입.csv": "CH"
}

# ==========================================
# 2. 고도화된 카테고리 구조 (Taxonomy 2.2)
# ==========================================
category_structure = {
    "피부 고민 해결": {
        "트러블/모공": ["여드름", "좁쌀", "트러블", "뾰루지", "모공", "요철", "블랙헤드", "화이트헤드", "흉터", "흔적", "화농성", "올라옴"],
        "진정/장벽": ["진정", "가라앉", "붉은기", "홍조", "재생", "회복", "장벽", "손상", "보호", "예민", "열감", "쿨링", "편안"],
        "미백/광채": ["미백", "밝아", "환해", "광채", "물광", "속광", "톤업", "잡티", "기미", "투명", "칙칙", "결 개선", "생기"]
    },
    "수분/유분 밸런스": {
        "보습/속건조": ["속건조", "속당김", "당김", "건조", "보습", "촉촉", "수분감", "푸석", "겉돌지", "수분충전", "촘촘", "속보습", "수부지"],
        "유분/피지": ["유분", "기름", "개기름", "번들", "산뜻", "피지", "보송", "오일리", "유분기", "답답함", "기름기"]
    },
    "제형/사용감": {
        "흡수/발림성": ["흡수", "스며", "발림", "제형", "질감", "밀착", "부드럽", "실키", "가벼운", "무거운", "점성", "묽은", "쫀쫀"],
        "끈적임/밀림": ["끈적", "밀림", "밀려", "밀착력", "잔여감", "겉돎", "유분감"]
    },
    "피부 자극": {
        "순함/저자극": ["순함", "순해", "순한", "자극없", "안심", "민감", "무자극", "따갑지", "순하게", "부담없는"],
        "자극/부작용": ["자극", "따갑", "따끔", "가려", "화끈", "뒤집", "붉어짐", "화끈거림", "가렵"]
    },
    "구매 가치/편의": {
        "가성비/가격": ["가격", "가성비", "세일", "혜택", "비쌈", "저렴", "대용량", "구성", "증정", "가심비", "착한", "착하다", "혜자", "이득", "사악"],
        "추천/재구매": ["추천", "강추", "인생템", "정착", "재구매", "지인", "비추", "선물", "만족", "재구입", "공병", "애정템"],
        "상황/편의": ["화장", "메이크업", "화잘먹", "펌프", "냄새", "향", "데일리", "팩토", "닦토", "흡토", "분사"]
    }
}

# ==========================================
# 3. [통합] 감성 사전 및 문맥 차단 로직
# ==========================================
SUPER_POS = ["인생템", "강추", "최고", "정착", "성공", "재구매", "무조건", "대박", "완전 추천", "효과 확실", "공병템", "확실", "아주 좋"]
NORMAL_POS = ["좋", "만족", "순해", "인정", "찰떡", "촉촉", "진정", "개선", "효과", "괜찮", "편안", "매끈", "맑", "탄탄", "믿고", "추천", "착한", "착하다", "혜자", "이득", "쫀쫀", "잘먹"]
SUPER_NEG = ["비추", "돈아깝", "거름", "최악", "다신안", "환불", "반품", "쓰레기", "뒤집어짐", "실망", "속았", "사악"]
NORMAL_NEG = ["아쉬", "안맞", "따가", "비싸", "밀려", "자극", "가려", "좁쌀", "번들", "답답", "냄새", "부족", "별로", "그다지", "불편", "심하", "의심", "부담되는", "들뜸", "뜸"]
INTENSIFIERS = ["너무", "진짜", "완전", "엄청", "진심", "정말", "되게", "매우", "솔직히", "겁나", "아주"]

ABSOLUTE_POS_PHRASES = ["화잘먹", "공병", "정착", "인생", "득템", "잘 먹어", "잘 먹음"]
QUOTE_MARKERS = ["광고", "바이럴", "영상", "유튜브", "남들은", "후기", "기대", "없어졌다고"]
CONTRAST_MARKERS = ["했는데", "했으나", "그치만", "하지만"]

NEGATIVE_REVERSAL = ["없", "않", "적", "사라", "진정", "개선", "나아", "제로", "줄어", "잡혔", "안 나", "안 뜨", "들어갔"]
NEG_TARGETS = ["자극", "끈적", "트러블", "밀림", "따가", "붉은기", "번들", "건조", "당김", "뒤집", "들뜸", "뜸", "유분"]

def smart_split(text):
    text = str(text)
    initial_splits = re.split(r'[.!\?\n]+', text)
    refined = []
    for s in initial_splits:
        s = s.strip()
        if len(s) > 35: # 문장이 길면 '고/서/데/며' 기준으로 추가 분리
            sub = re.split(r'(?<=[고서데며])\s+', s)
            refined.extend(sub)
        else:
            refined.append(s)
    return [s.strip() for s in refined if len(s.strip()) > 2]

def calculate_refined_sentiment(sentence, matched_kws):
    score = 0
    # 0. 광고/인용구 필터링 (베이스의 핵심 로직 반영)
    is_quote = any(q in sentence for q in QUOTE_MARKERS)
    has_contrast = any(c in sentence for c in CONTRAST_MARKERS)

    # 1. 기본 점수
    for wp in SUPER_POS: score += 3 if wp in sentence else 0
    for wp in NORMAL_POS: score += 1 if wp in sentence else 0
    for wn in SUPER_NEG: score -= 3 if wn in sentence else 0
    for wn in NORMAL_NEG: score -= 1 if wn in sentence else 0

    # 2. 절대 긍정 및 강조어 처리 (세럼 로직 반영)
    if any(ap in sentence for ap in ABSOLUTE_POS_PHRASES): score += 2
    if any(i in sentence for i in INTENSIFIERS):
        if score > 0: score += 1
        elif score < 0: score -= 1

    # 3. 부정 요소 반전 및 악화 로직
    for target in NEG_TARGETS:
        if target in sentence:
            after_target = sentence.split(target)[-1]
            # 부정어가 해소된 경우 (긍정 처리)
            if not is_quote and any(rev in after_target for rev in NEGATIVE_REVERSAL):
                score = abs(score) + 2
            # 부정어가 심해진 경우 (강력 감점)
            if any(neg in after_target for neg in ["올라", "나왔", "생겼", "심해", "안 좋아"]):
                score -= 3

    # 4. 광고/기대 대조군 강제 보정
    if is_quote and has_contrast:
        if any(n in sentence for n in NORMAL_NEG + SUPER_NEG + ["올라", "안 좋아", "나왔"]):
            score = -4

    # 5. 가격 특화 처리
    if "가격" in sentence or "가성비" in sentence:
        if any(p in sentence for p in ["착한", "착하다", "혜자", "부담없", "저렴", "이득"]): score = max(score, 2)
        if any(n in sentence for n in ["사악", "비싸", "부담되"]): score = min(score, -2)

    score = int(max(-5, min(5, score)))
    label = "긍정" if score > 0 else "부정" if score < 0 else "보통"
    return label, score

# ==========================================
# 4. 분석 실행 및 리포트 저장
# ==========================================
final_rows = []

for f, p_code in product_code_map.items():
    if not os.path.exists(f): continue
    df = pd.read_csv(f)
    p_name = f.split('.')[-2].strip() if '.' in f else f
    df['날짜'] = df['날짜'].astype(str)
    df = df.sort_values('날짜')
    date_id_counts = {}

    for _, row in df.iterrows():
        date_raw = row['날짜']
        date_clean = date_raw.replace('.', '')[2:8]
        date_id_counts[date_clean] = date_id_counts.get(date_clean, 0) + 1
        review_id = f"{p_code}_{date_clean}_{date_id_counts[date_clean]:03d}"

        raw_text = str(row.get('clean_text', row.get('리뷰내용', '')))
        sentences = smart_split(raw_text)

        review_sentences_data = []
        review_all_kws, review_major_cats = set(), set()

        for sent in sentences:
            sent_major, sent_minor, sent_kws = set(), set(), set()
            for major, submap in category_structure.items():
                for minor, kws in submap.items():
                    matched = [kw for kw in kws if kw in sent]
                    if matched:
                        sent_major.add(major); sent_minor.add(minor); sent_kws.update(matched)
                        review_major_cats.add(major); review_all_kws.update(matched)

            if sent_kws:
                label, score = calculate_refined_sentiment(sent, sent_kws)
                review_sentences_data.append({"문장": sent, "라벨": label, "점수": score, "대": sent_major, "소": sent_minor, "키": sent_kws})

        avg_score = round(sum(d["점수"] for d in review_sentences_data) / len(review_sentences_data), 1) if review_sentences_data else 0.0

        for res in review_sentences_data:
            final_rows.append({
                "리뷰ID": review_id,
                "제품군": "로션/밀크/크림",
                "제품명": p_name,
                "날짜": date_raw,
                "가격": row.get('price', row.get('가격', '-')),
                "스킨타입": row.get('스킨타입', '-'),
                "태그": row.get('태그', '-'),
                "대분류": ", ".join(sorted(res["대"])),
                "소분류": ", ".join(sorted(res["소"])),
                "정보밀도(대분류수)": len(review_major_cats),
                "별점": row.get('별점', '-'),
                "긍부정": res["라벨"],
                "감성점수": res["점수"],
                "매칭키워드": ", ".join(sorted(res["키"])),
                "추출문장": res["문장"],
                "clean_text": raw_text,
                "리뷰_평균점수": avg_score,
                "전체_매칭키워드": ", ".join(sorted(review_all_kws))
            })

output_name = "lotion_milk_cream_analysis_report.csv"
pd.DataFrame(final_rows).to_csv(output_name, index=False, encoding='utf-8-sig')
print(f"✔ 분석 완료: {output_name}")

✔ 분석 완료: lotion_milk_cream_analysis_report.csv


In [15]:
import pandas as pd
import numpy as np
import os

# 1. 원본 데이터 로드 (기존 문장 단위 분석 결과 파일)
base_file = "lotion_milk_cream_analysis_report.csv"

if not os.path.exists(base_file):
    print(f"⚠ {base_file} 파일이 없습니다. 분석을 먼저 실행해주세요.")
else:
    df = pd.read_csv(base_file)

    # 2. 속성(스킨타입 + 태그) 통합 및 분리 함수 (정보없음 포함 로직)
    def get_attribute_df(dataframe):
        temp_df = dataframe.copy()

        def combine_attrs(row):
            s = str(row.get('스킨타입', '-')).strip()
            t = str(row.get('태그', '-')).strip()

            # 비어있거나 의미 없는 값 체크
            def is_empty(val):
                return val in ['-', 'nan', 'None', '', 'nan ']

            attrs = []
            if not is_empty(s):
                attrs.extend([item.strip() for item in s.split(",")])
            if not is_empty(t):
                attrs.extend([item.strip() for item in t.split(",")])

            # 둘 다 정보가 없는 경우에만 '정보없음' 할당
            if not attrs:
                return ["정보없음"]
            # 중복 제거 (스킨타입과 태그에 중복된 단어가 있을 경우 대비)
            return list(set(attrs))

        temp_df['Attribute'] = temp_df.apply(combine_attrs, axis=1)
        # 리스트로 묶인 속성들을 행별로 분리(explode)
        temp_df = temp_df.explode('Attribute')
        return temp_df

    attr_df = get_attribute_df(df)
    final_report_rows = []
    products = df['제품명'].unique()

    for product in products:
        p_df = df[df['제품명'] == product]
        p_attr_df = attr_df[attr_df['제품명'] == product]

        # --- PART A: 속성별 수치 요약 (정보없음 포함) ---
        # 가나다 순 정렬 (정보없음이 포함됨)
        attributes = sorted(p_attr_df['Attribute'].unique())
        for attr in attributes:
            group = p_attr_df[p_attr_df['Attribute'] == attr]
            user_count = group['리뷰ID'].nunique()
            avg_score = round(group['감성점수'].mean(), 2)

            # 소분류(Sub-category) 추출 및 집계
            subcat_df = group.copy()
            subcat_df['Subcat'] = subcat_df['소분류'].astype(str).str.split(", ")
            subcat_df = subcat_df.explode('Subcat')
            subcat_df = subcat_df[subcat_df['Subcat'] != 'nan']

            pos_subs = subcat_df[subcat_df['긍부정'] == '긍정']['Subcat'].value_counts().head(5)
            pos_str = "\n".join([f"{k} ({v})" for k, v in pos_subs.items()]) if not pos_subs.empty else "-"

            neg_subs = subcat_df[subcat_df['긍부정'] == '부정']['Subcat'].value_counts().head(5)
            neg_str = "\n".join([f"{k} ({v})" for k, v in neg_subs.items()]) if not neg_subs.empty else "-"

            final_report_rows.append({
                "제품명": product,
                "상세 항목 및 요약": attr,
                "사용자 수 / 정보밀도": f"{user_count}명",
                "긍정 소분류 (Top 5)": pos_str,
                "부정 소분류 (Top 5)": neg_str,
                "종합 만족도": "매우만족" if avg_score >= 1.5 else "만족" if avg_score >= 0.5 else "보통",
                "평균 점수": avg_score
            })

        # --- PART B: 제품 총평 랩업 (Wrap-up) ---
        # 속성별 유니크 유저 수 집계
        demo_counts = p_attr_df.groupby('Attribute')['리뷰ID'].nunique().sort_values(ascending=False).to_dict()
        demo_str = ", ".join([f"{k} {v}명" for k, v in demo_counts.items()])

        # 제품 전체에서 가장 많이 언급된 특징과 감성
        overall_subcat_df = p_df.copy()
        overall_subcat_df['Subcat'] = overall_subcat_df['소분류'].astype(str).str.split(", ")
        overall_subcat_df = overall_subcat_df.explode('Subcat')
        overall_subcat_df = overall_subcat_df[overall_subcat_df['Subcat'] != 'nan']

        if not overall_subcat_df.empty:
            top_sub = overall_subcat_df['Subcat'].value_counts().idxmax()
            top_sub_score = overall_subcat_df[overall_subcat_df['Subcat'] == top_sub]['감성점수'].mean()
            top_sub_sent = "긍정" if top_sub_score > 0 else "부정"
        else:
            top_sub = "정보없음"
            top_sub_sent = "보통"

        wrap_up_text = f"[{product} 랩업] 참여 유저: ({demo_str}). 이 제품에서 가장 많이 언급된 특징은 '{top_sub}'(으)로, 이에 대해 주로 {top_sub_sent}적인 반응을 보였습니다."

        final_report_rows.append({
            "제품명": product,
            "상세 항목 및 요약": wrap_up_text,
            "사용자 수 / 정보밀도": f"총 {p_df['리뷰ID'].nunique()}명",
            "긍정 소분류 (Top 5)": "★★ 제품 전체 요약 ★★",
            "부정 소분류 (Top 5)": "★★ 제품 전체 요약 ★★",
            "종합 만족도": "-",
            "평균 점수": round(p_df['감성점수'].mean(), 2)
        })

        # --- PART C: 정보 밀도 높은 대표 리뷰 TOP 3 ---
        # 중복 제거 후 정보밀도 순으로 정렬
        top_reviews = p_df.drop_duplicates(subset='리뷰ID').sort_values(by='정보밀도(대분류수)', ascending=False).head(3)

        for i, (_, r_row) in enumerate(top_reviews.iterrows()):
            final_report_rows.append({
                "제품명": product,
                "상세 항목 및 요약": f"[대표리뷰 {i+1}] {r_row['clean_text']}",
                "사용자 수 / 정보밀도": f"밀도: {r_row['정보밀도(대분류수)']}",
                # [수정] 작성자 정보 자리에 고유 리뷰ID를 표시합니다.
                "긍정 소분류 (Top 5)": f"리뷰ID: {r_row['리뷰ID']}",
                "부정 소분류 (Top 5)": f"태그: {r_row['태그']}",
                "종합 만족도": f"별점: {r_row['별점']}",
                "평균 점수": r_row['리뷰_평균점수']
            })

    # 최종 결과 저장 (UTF-8-SIG 인코딩으로 엑셀 한글 깨짐 방지)
    final_df = pd.DataFrame(final_report_rows)
    final_df.to_csv("lotion_milk_cream_summary_report.csv", index=False, encoding='utf-8-sig')
    print("✔ 소비자 인사이트 요약 리포트 생성 완료: lotion_milk_cream_summary_report.csv")

✔ 소비자 인사이트 요약 리포트 생성 완료: lotion_milk_cream_summary_report.csv


### 토너

In [16]:
import pandas as pd
import os
import re

# ==========================================
# 1. 제품별 식별 코드 매핑 (토너 라인업)
# ==========================================
product_code_map = {
    "reviews_cleaned_24. 라이스 70 글로우 밀키 토너 250ml.csv": "TR",
    "reviews_cleaned_23. 복숭아 나이아신아마이드 에센스 토너 250mL.csv": "TP"
}

# ==========================================
# 2. 고도화된 카테고리 구조 (Taxonomy 2.2)
# ==========================================
category_structure = {
    "피부 고민 해결": {
        "트러블/모공": ["여드름", "좁쌀", "트러블", "뾰루지", "모공", "요철", "블랙헤드", "화이트헤드", "흉터", "흔적", "화농성", "올라옴"],
        "진정/장벽": ["진정", "가라앉", "붉은기", "홍조", "재생", "회복", "장벽", "손상", "보호", "예민", "열감", "쿨링", "편안"],
        "미백/광채": ["미백", "밝아", "환해", "광채", "물광", "속광", "톤업", "잡티", "기미", "투명", "칙칙", "결 개선", "생기"]
    },
    "수분/유분 밸런스": {
        "보습/속건조": ["속건조", "속당김", "당김", "건조", "보습", "촉촉", "수분감", "푸석", "겉돌지", "수분충전", "촘촘", "속보습", "수부지"],
        "유분/피지": ["유분", "기름", "개기름", "번들", "산뜻", "피지", "보송", "오일리", "유분기", "답답함", "기름기"]
    },
    "제형/사용감": {
        "흡수/발림성": ["흡수", "스며", "발림", "제형", "질감", "밀착", "부드럽", "실키", "가벼운", "무거운", "점성", "묽은", "쫀쫀"],
        "끈적임/밀림": ["끈적", "밀림", "밀려", "밀착력", "잔여감", "겉돎", "유분감"]
    },
    "피부 자극": {
        "순함/저자극": ["순함", "순해", "순한", "자극없", "안심", "민감", "무자극", "따갑지", "순하게", "부담없는"],
        "자극/부작용": ["자극", "따갑", "따끔", "가려", "화끈", "뒤집", "붉어짐", "화끈거림", "가렵"]
    },
    "구매 가치/편의": {
        "가성비/가격": ["가격", "가성비", "세일", "혜택", "비쌈", "저렴", "대용량", "구성", "증정", "가심비", "착한", "착하다", "혜자", "이득", "사악"],
        "추천/재구매": ["추천", "강추", "인생템", "정착", "재구매", "지인", "비추", "선물", "만족", "재구입", "공병", "애정템"],
        "상황/편의": ["화장", "메이크업", "화잘먹", "펌프", "냄새", "향", "데일리", "팩토", "닦토", "흡토", "분사"]
    }
}

# ==========================================
# 3. [통합] 감성 사전 및 문맥 차단 로직
# ==========================================
SUPER_POS = ["인생템", "강추", "최고", "정착", "성공", "재구매", "무조건", "대박", "완전 추천", "효과 확실", "공병템", "확실", "아주 좋"]
NORMAL_POS = ["좋", "만족", "순해", "인정", "찰떡", "촉촉", "진정", "개선", "효과", "괜찮", "편안", "매끈", "맑", "탄탄", "믿고", "추천", "착한", "착하다", "혜자", "이득", "쫀쫀", "잘먹"]
SUPER_NEG = ["비추", "돈아깝", "거름", "최악", "다신안", "환불", "반품", "쓰레기", "뒤집어짐", "실망", "속았", "사악"]
NORMAL_NEG = ["아쉬", "안맞", "따가", "비싸", "밀려", "자극", "가려", "좁쌀", "번들", "답답", "냄새", "부족", "별로", "그다지", "불편", "심하", "의심", "부담되는", "들뜸", "뜸"]
INTENSIFIERS = ["너무", "진짜", "완전", "엄청", "진심", "정말", "되게", "매우", "솔직히", "겁나", "아주"]

ABSOLUTE_POS_PHRASES = ["화잘먹", "공병", "정착", "인생", "득템", "잘 먹어", "잘 먹음"]
QUOTE_MARKERS = ["광고", "바이럴", "영상", "유튜브", "남들은", "후기", "기대", "없어졌다고"]
CONTRAST_MARKERS = ["했는데", "했으나", "그치만", "하지만"]

NEGATIVE_REVERSAL = ["없", "않", "적", "사라", "진정", "개선", "나아", "제로", "줄어", "잡혔", "안 나", "안 뜨", "들어갔"]
NEG_TARGETS = ["자극", "끈적", "트러블", "밀림", "따가", "붉은기", "번들", "건조", "당김", "뒤집", "들뜸", "뜸", "유분"]

def smart_split(text):
    text = str(text)
    initial_splits = re.split(r'[.!\?\n]+', text)
    refined = []
    for s in initial_splits:
        s = s.strip()
        if len(s) > 35:
            sub = re.split(r'(?<=[고서데며])\s+', s)
            refined.extend(sub)
        else:
            refined.append(s)
    return [s.strip() for s in refined if len(s.strip()) > 2]

def calculate_refined_sentiment(sentence, matched_kws):
    score = 0
    is_quote = any(q in sentence for q in QUOTE_MARKERS)
    has_contrast = any(c in sentence for c in CONTRAST_MARKERS)

    for wp in SUPER_POS: score += 3 if wp in sentence else 0
    for wp in NORMAL_POS: score += 1 if wp in sentence else 0
    for wn in SUPER_NEG: score -= 3 if wn in sentence else 0
    for wn in NORMAL_NEG: score -= 1 if wn in sentence else 0

    if any(ap in sentence for ap in ABSOLUTE_POS_PHRASES):
        score += 2

    if any(i in sentence for i in INTENSIFIERS):
        if score > 0: score += 1
        elif score < 0: score -= 1

    for target in NEG_TARGETS:
        if target in sentence:
            after_target = sentence.split(target)[-1]
            if not is_quote and any(rev in after_target for rev in NEGATIVE_REVERSAL):
                score = abs(score) + 2
            if any(neg in after_target for neg in ["올라", "나왔", "생겼", "심해", "안 좋아"]):
                score -= 3

    if is_quote and has_contrast:
        if any(n in sentence for n in NORMAL_NEG + SUPER_NEG + ["올라", "안 좋아", "나왔"]):
            score = -4

    if "가격" in sentence or "가성비" in sentence:
        if any(p in sentence for p in ["착한", "착하다", "혜자", "부담없", "저렴", "이득"]):
            score = max(score, 2)
        if any(n in sentence for n in ["사악", "비싸", "부담되"]):
            score = min(score, -2)

    score = int(max(-5, min(5, score)))
    label = "긍정" if score > 0 else "부정" if score < 0 else "보통"
    return label, score

# ==========================================
# 4. 분석 실행 및 리포트 저장
# ==========================================
final_rows = []

for f, p_code in product_code_map.items():
    if not os.path.exists(f): continue
    df = pd.read_csv(f)
    p_name = f.split('.')[-2].strip() if '.' in f else f
    df['날짜'] = df['날짜'].astype(str)
    df = df.sort_values('날짜')
    date_id_counts = {}

    for _, row in df.iterrows():
        date_raw = row['날짜']
        date_clean = date_raw.replace('.', '')[2:8]
        date_id_counts[date_clean] = date_id_counts.get(date_clean, 0) + 1

        review_id = f"{p_code}_{date_clean}_{date_id_counts[date_clean]:03d}"
        raw_text = str(row.get('clean_text', row.get('리뷰내용', '')))
        sentences = smart_split(raw_text)

        review_sentences_data = []
        review_all_kws, review_major_cats = set(), set()

        for sent in sentences:
            sent_major, sent_minor, sent_kws = set(), set(), set()
            for major, submap in category_structure.items():
                for minor, kws in submap.items():
                    matched = [kw for kw in kws if kw in sent]
                    if matched:
                        sent_major.add(major); sent_minor.add(minor); sent_kws.update(matched)
                        review_major_cats.add(major); review_all_kws.update(matched)

            if sent_kws:
                label, score = calculate_refined_sentiment(sent, sent_kws)
                review_sentences_data.append({"문장": sent, "라벨": label, "점수": score, "대": sent_major, "소": sent_minor, "키": sent_kws})

        avg_score = round(sum(d["점수"] for d in review_sentences_data) / len(review_sentences_data), 1) if review_sentences_data else 0.0

        for res in review_sentences_data:
            final_rows.append({
                "리뷰ID": review_id,
                "제품군": "토너",
                "제품명": p_name,
                "날짜": date_raw,
                "가격": row.get('price', row.get('가격', '-')),
                "스킨타입": row.get('스킨타입', '-'),
                "태그": row.get('태그', '-'),
                "대분류": ", ".join(sorted(res["대"])),
                "소분류": ", ".join(sorted(res["소"])),
                "정보밀도(대분류수)": len(review_major_cats),
                "별점": row.get('별점', '-'),
                "긍부정": res["라벨"],
                "감성점수": res["점수"],
                "매칭키워드": ", ".join(sorted(res["키"])),
                "추출문장": res["문장"],
                "clean_text": raw_text,
                "리뷰_평균점수": avg_score,
                "전체_매칭키워드": ", ".join(sorted(review_all_kws))
            })

output_file = "toner_analysis_report.csv"
pd.DataFrame(final_rows).to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"✔ 분석 완료: {output_file}")

✔ 분석 완료: toner_analysis_report.csv


In [17]:
import pandas as pd
import numpy as np
import os

# 1. 원본 데이터 로드 (기존 문장 단위 분석 결과 파일)
base_file = "toner_analysis_report.csv"

if not os.path.exists(base_file):
    print(f"⚠ {base_file} 파일이 없습니다. 분석을 먼저 실행해주세요.")
else:
    df = pd.read_csv(base_file)

    # 2. 속성(스킨타입 + 태그) 통합 및 분리 함수 (정보없음 포함 로직)
    def get_attribute_df(dataframe):
        temp_df = dataframe.copy()

        def combine_attrs(row):
            s = str(row.get('스킨타입', '-')).strip()
            t = str(row.get('태그', '-')).strip()

            # 비어있거나 의미 없는 값 체크
            def is_empty(val):
                return val in ['-', 'nan', 'None', '', 'nan ']

            attrs = []
            if not is_empty(s):
                attrs.extend([item.strip() for item in s.split(",")])
            if not is_empty(t):
                attrs.extend([item.strip() for item in t.split(",")])

            # 둘 다 정보가 없는 경우에만 '정보없음' 할당
            if not attrs:
                return ["정보없음"]
            # 중복 제거 (스킨타입과 태그에 중복된 단어가 있을 경우 대비)
            return list(set(attrs))

        temp_df['Attribute'] = temp_df.apply(combine_attrs, axis=1)
        # 리스트로 묶인 속성들을 행별로 분리(explode)
        temp_df = temp_df.explode('Attribute')
        return temp_df

    attr_df = get_attribute_df(df)
    final_report_rows = []
    products = df['제품명'].unique()

    for product in products:
        p_df = df[df['제품명'] == product]
        p_attr_df = attr_df[attr_df['제품명'] == product]

        # --- PART A: 속성별 수치 요약 (정보없음 포함) ---
        # 가나다 순 정렬 (정보없음이 포함됨)
        attributes = sorted(p_attr_df['Attribute'].unique())
        for attr in attributes:
            group = p_attr_df[p_attr_df['Attribute'] == attr]
            user_count = group['리뷰ID'].nunique()
            avg_score = round(group['감성점수'].mean(), 2)

            # 소분류(Sub-category) 추출 및 집계
            subcat_df = group.copy()
            subcat_df['Subcat'] = subcat_df['소분류'].astype(str).str.split(", ")
            subcat_df = subcat_df.explode('Subcat')
            subcat_df = subcat_df[subcat_df['Subcat'] != 'nan']

            pos_subs = subcat_df[subcat_df['긍부정'] == '긍정']['Subcat'].value_counts().head(5)
            pos_str = "\n".join([f"{k} ({v})" for k, v in pos_subs.items()]) if not pos_subs.empty else "-"

            neg_subs = subcat_df[subcat_df['긍부정'] == '부정']['Subcat'].value_counts().head(5)
            neg_str = "\n".join([f"{k} ({v})" for k, v in neg_subs.items()]) if not neg_subs.empty else "-"

            final_report_rows.append({
                "제품명": product,
                "상세 항목 및 요약": attr,
                "사용자 수 / 정보밀도": f"{user_count}명",
                "긍정 소분류 (Top 5)": pos_str,
                "부정 소분류 (Top 5)": neg_str,
                "종합 만족도": "매우만족" if avg_score >= 1.5 else "만족" if avg_score >= 0.5 else "보통",
                "평균 점수": avg_score
            })

        # --- PART B: 제품 총평 랩업 (Wrap-up) ---
        # 속성별 유니크 유저 수 집계
        demo_counts = p_attr_df.groupby('Attribute')['리뷰ID'].nunique().sort_values(ascending=False).to_dict()
        demo_str = ", ".join([f"{k} {v}명" for k, v in demo_counts.items()])

        # 제품 전체에서 가장 많이 언급된 특징과 감성
        overall_subcat_df = p_df.copy()
        overall_subcat_df['Subcat'] = overall_subcat_df['소분류'].astype(str).str.split(", ")
        overall_subcat_df = overall_subcat_df.explode('Subcat')
        overall_subcat_df = overall_subcat_df[overall_subcat_df['Subcat'] != 'nan']

        if not overall_subcat_df.empty:
            top_sub = overall_subcat_df['Subcat'].value_counts().idxmax()
            top_sub_score = overall_subcat_df[overall_subcat_df['Subcat'] == top_sub]['감성점수'].mean()
            top_sub_sent = "긍정" if top_sub_score > 0 else "부정"
        else:
            top_sub = "정보없음"
            top_sub_sent = "보통"

        wrap_up_text = f"[{product} 랩업] 참여 유저: ({demo_str}). 이 제품에서 가장 많이 언급된 특징은 '{top_sub}'(으)로, 이에 대해 주로 {top_sub_sent}적인 반응을 보였습니다."

        final_report_rows.append({
            "제품명": product,
            "상세 항목 및 요약": wrap_up_text,
            "사용자 수 / 정보밀도": f"총 {p_df['리뷰ID'].nunique()}명",
            "긍정 소분류 (Top 5)": "★★ 제품 전체 요약 ★★",
            "부정 소분류 (Top 5)": "★★ 제품 전체 요약 ★★",
            "종합 만족도": "-",
            "평균 점수": round(p_df['감성점수'].mean(), 2)
        })

        # --- PART C: 정보 밀도 높은 대표 리뷰 TOP 3 ---
        # 중복 제거 후 정보밀도 순으로 정렬
        top_reviews = p_df.drop_duplicates(subset='리뷰ID').sort_values(by='정보밀도(대분류수)', ascending=False).head(3)

        for i, (_, r_row) in enumerate(top_reviews.iterrows()):
            final_report_rows.append({
                "제품명": product,
                "상세 항목 및 요약": f"[대표리뷰 {i+1}] {r_row['clean_text']}",
                "사용자 수 / 정보밀도": f"밀도: {r_row['정보밀도(대분류수)']}",
                # [수정] 작성자 정보 자리에 고유 리뷰ID를 표시합니다.
                "긍정 소분류 (Top 5)": f"리뷰ID: {r_row['리뷰ID']}",
                "부정 소분류 (Top 5)": f"태그: {r_row['태그']}",
                "종합 만족도": f"별점: {r_row['별점']}",
                "평균 점수": r_row['리뷰_평균점수']
            })

    # 최종 결과 저장 (UTF-8-SIG 인코딩으로 엑셀 한글 깨짐 방지)
    final_df = pd.DataFrame(final_report_rows)
    final_df.to_csv("toner_analysis_summary_report.csv", index=False, encoding='utf-8-sig')
    print("✔ 소비자 인사이트 요약 리포트 생성 완료: toner_analysis_summary_report.csv")

✔ 소비자 인사이트 요약 리포트 생성 완료: toner_analysis_summary_report.csv


### 앰플

In [18]:
import pandas as pd
import os
import re

# ==========================================
# 1. 제품별 식별 코드 매핑 (앰플 라인업)
# ==========================================
product_code_map = {
    "reviews_cleaned_21. 어성초 80 수분 진정 앰플 30ml.csv": "AH"
}

# ==========================================
# 2. 고도화된 카테고리 구조 (Taxonomy 2.2)
# ==========================================
category_structure = {
    "피부 고민 해결": {
        "트러블/모공": ["여드름", "좁쌀", "트러블", "뾰루지", "모공", "요철", "블랙헤드", "화이트헤드", "흉터", "흔적", "화농성", "올라옴"],
        "진정/장벽": ["진정", "가라앉", "붉은기", "홍조", "재생", "회복", "장벽", "손상", "보호", "예민", "열감", "쿨링", "편안"],
        "미백/광채": ["미백", "밝아", "환해", "광채", "물광", "속광", "톤업", "잡티", "기미", "투명", "칙칙", "결 개선", "생기"]
    },
    "수분/유분 밸런스": {
        "보습/속건조": ["속건조", "속당김", "당김", "건조", "보습", "촉촉", "수분감", "푸석", "겉돌지", "수분충전", "촘촘", "속보습", "수부지"],
        "유분/피지": ["유분", "기름", "개기름", "번들", "산뜻", "피지", "보송", "오일리", "유분기", "답답함", "기름기"]
    },
    "제형/사용감": {
        "흡수/발림성": ["흡수", "스며", "발림", "제형", "질감", "밀착", "부드럽", "실키", "가벼운", "무거운", "점성", "묽은", "쫀쫀"],
        "끈적임/밀림": ["끈적", "밀림", "밀려", "밀착력", "잔여감", "겉돎", "유분감"]
    },
    "피부 자극": {
        "순함/저자극": ["순함", "순해", "순한", "자극없", "안심", "민감", "무자극", "따갑지", "순하게", "부담없는"],
        "자극/부작용": ["자극", "따갑", "따끔", "가려", "화끈", "뒤집", "붉어짐", "화끈거림", "가렵"]
    },
    "구매 가치/편의": {
        "가성비/가격": ["가격", "가성비", "세일", "혜택", "비쌈", "저렴", "대용량", "구성", "증정", "가심비", "착한", "착하다", "혜자", "이득", "사악"],
        "추천/재구매": ["추천", "강추", "인생템", "정착", "재구매", "지인", "비추", "선물", "만족", "재구입", "공병", "애정템"],
        "상황/편의": ["화장", "메이크업", "화잘먹", "펌프", "냄새", "향", "데일리", "팩토", "닦토", "흡토", "분사"]
    }
}

# ==========================================
# 3. [통합] 감성 사전 및 문맥 차단 로직
# ==========================================
SUPER_POS = ["인생템", "강추", "최고", "정착", "성공", "재구매", "무조건", "대박", "완전 추천", "효과 확실", "공병템", "확실", "아주 좋"]
NORMAL_POS = ["좋", "만족", "순해", "인정", "찰떡", "촉촉", "진정", "개선", "효과", "괜찮", "편안", "매끈", "맑", "탄탄", "믿고", "추천", "착한", "착하다", "혜자", "이득", "쫀쫀", "잘먹"]
SUPER_NEG = ["비추", "돈아깝", "거름", "최악", "다신안", "환불", "반품", "쓰레기", "뒤집어짐", "실망", "속았", "사악"]
NORMAL_NEG = ["아쉬", "안맞", "따가", "비싸", "밀려", "자극", "가려", "좁쌀", "번들", "답답", "냄새", "부족", "별로", "그다지", "불편", "심하", "의심", "부담되는", "들뜸", "뜸"]
INTENSIFIERS = ["너무", "진짜", "완전", "엄청", "진심", "정말", "되게", "매우", "솔직히", "겁나", "아주"]

ABSOLUTE_POS_PHRASES = ["화잘먹", "공병", "정착", "인생", "득템", "잘 먹어", "잘 먹음"]
QUOTE_MARKERS = ["광고", "바이럴", "영상", "유튜브", "남들은", "후기", "기대", "없어졌다고"]
CONTRAST_MARKERS = ["했는데", "했으나", "그치만", "하지만"]

NEGATIVE_REVERSAL = ["없", "않", "적", "사라", "진정", "개선", "나아", "제로", "줄어", "잡혔", "안 나", "안 뜨", "들어갔"]
NEG_TARGETS = ["자극", "끈적", "트러블", "밀림", "따가", "붉은기", "번들", "건조", "당김", "뒤집", "들뜸", "뜸", "유분"]

def smart_split(text):
    text = str(text)
    initial_splits = re.split(r'[.!\?\n]+', text)
    refined = []
    for s in initial_splits:
        s = s.strip()
        if len(s) > 35:
            sub = re.split(r'(?<=[고서데며])\s+', s)
            refined.extend(sub)
        else:
            refined.append(s)
    return [s.strip() for s in refined if len(s.strip()) > 2]

def calculate_refined_sentiment(sentence, matched_kws):
    score = 0
    is_quote = any(q in sentence for q in QUOTE_MARKERS)
    has_contrast = any(c in sentence for c in CONTRAST_MARKERS)

    for wp in SUPER_POS: score += 3 if wp in sentence else 0
    for wp in NORMAL_POS: score += 1 if wp in sentence else 0
    for wn in SUPER_NEG: score -= 3 if wn in sentence else 0
    for wn in NORMAL_NEG: score -= 1 if wn in sentence else 0

    if any(ap in sentence for ap in ABSOLUTE_POS_PHRASES):
        score += 2

    if any(i in sentence for i in INTENSIFIERS):
        if score > 0: score += 1
        elif score < 0: score -= 1

    for target in NEG_TARGETS:
        if target in sentence:
            after_target = sentence.split(target)[-1]
            if not is_quote and any(rev in after_target for rev in NEGATIVE_REVERSAL):
                score = abs(score) + 2
            if any(neg in after_target for neg in ["올라", "나왔", "생겼", "심해", "안 좋아"]):
                score -= 3

    if is_quote and has_contrast:
        if any(n in sentence for n in NORMAL_NEG + SUPER_NEG + ["올라", "안 좋아", "나왔"]):
            score = -4

    if "가격" in sentence or "가성비" in sentence:
        if any(p in sentence for p in ["착한", "착하다", "혜자", "부담없", "저렴", "이득"]):
            score = max(score, 2)
        if any(n in sentence for n in ["사악", "비싸", "부담되"]):
            score = min(score, -2)

    score = int(max(-5, min(5, score)))
    label = "긍정" if score > 0 else "부정" if score < 0 else "보통"
    return label, score

# ==========================================
# 4. 분석 실행 및 리포트 저장
# ==========================================
final_rows = []

for f, p_code in product_code_map.items():
    if not os.path.exists(f): continue
    df = pd.read_csv(f)
    p_name = f.split('.')[-2].strip() if '.' in f else f
    df['날짜'] = df['날짜'].astype(str)
    df = df.sort_values('날짜')
    date_id_counts = {}

    for _, row in df.iterrows():
        date_raw = row['날짜']
        date_clean = date_raw.replace('.', '')[2:8]
        date_id_counts[date_clean] = date_id_counts.get(date_clean, 0) + 1

        review_id = f"{p_code}_{date_clean}_{date_id_counts[date_clean]:03d}"
        raw_text = str(row.get('clean_text', row.get('리뷰내용', '')))
        sentences = smart_split(raw_text)

        review_sentences_data = []
        review_all_kws, review_major_cats = set(), set()

        for sent in sentences:
            sent_major, sent_minor, sent_kws = set(), set(), set()
            for major, submap in category_structure.items():
                for minor, kws in submap.items():
                    matched = [kw for kw in kws if kw in sent]
                    if matched:
                        sent_major.add(major); sent_minor.add(minor); sent_kws.update(matched)
                        review_major_cats.add(major); review_all_kws.update(matched)

            if sent_kws:
                label, score = calculate_refined_sentiment(sent, sent_kws)
                review_sentences_data.append({"문장": sent, "라벨": label, "점수": score, "대": sent_major, "소": sent_minor, "키": sent_kws})

        avg_score = round(sum(d["점수"] for d in review_sentences_data) / len(review_sentences_data), 1) if review_sentences_data else 0.0

        for res in review_sentences_data:
            final_rows.append({
                "리뷰ID": review_id,
                "제품군": "앰플",
                "제품명": p_name,
                "날짜": date_raw,
                "가격": row.get('price', row.get('가격', '-')),
                "스킨타입": row.get('스킨타입', '-'),
                "태그": row.get('태그', '-'),
                "대분류": ", ".join(sorted(res["대"])),
                "소분류": ", ".join(sorted(res["소"])),
                "정보밀도(대분류수)": len(review_major_cats),
                "별점": row.get('별점', '-'),
                "긍부정": res["라벨"],
                "감성점수": res["점수"],
                "매칭키워드": ", ".join(sorted(res["키"])),
                "추출문장": res["문장"],
                "clean_text": raw_text,
                "리뷰_평균점수": avg_score,
                "전체_매칭키워드": ", ".join(sorted(review_all_kws))
            })

output_file = "ampoule_analysis_report.csv"
pd.DataFrame(final_rows).to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"✔ 분석 완료: {output_file}")

✔ 분석 완료: ampoule_analysis_report.csv


In [19]:
import pandas as pd
import numpy as np
import os

# 1. 원본 데이터 로드 (기존 문장 단위 분석 결과 파일)
base_file = "ampoule_analysis_report.csv"

if not os.path.exists(base_file):
    print(f"⚠ {base_file} 파일이 없습니다. 분석을 먼저 실행해주세요.")
else:
    df = pd.read_csv(base_file)

    # 2. 속성(스킨타입 + 태그) 통합 및 분리 함수 (정보없음 포함 로직)
    def get_attribute_df(dataframe):
        temp_df = dataframe.copy()

        def combine_attrs(row):
            s = str(row.get('스킨타입', '-')).strip()
            t = str(row.get('태그', '-')).strip()

            # 비어있거나 의미 없는 값 체크
            def is_empty(val):
                return val in ['-', 'nan', 'None', '', 'nan ']

            attrs = []
            if not is_empty(s):
                attrs.extend([item.strip() for item in s.split(",")])
            if not is_empty(t):
                attrs.extend([item.strip() for item in t.split(",")])

            # 둘 다 정보가 없는 경우에만 '정보없음' 할당
            if not attrs:
                return ["정보없음"]
            # 중복 제거 (스킨타입과 태그에 중복된 단어가 있을 경우 대비)
            return list(set(attrs))

        temp_df['Attribute'] = temp_df.apply(combine_attrs, axis=1)
        # 리스트로 묶인 속성들을 행별로 분리(explode)
        temp_df = temp_df.explode('Attribute')
        return temp_df

    attr_df = get_attribute_df(df)
    final_report_rows = []
    products = df['제품명'].unique()

    for product in products:
        p_df = df[df['제품명'] == product]
        p_attr_df = attr_df[attr_df['제품명'] == product]

        # --- PART A: 속성별 수치 요약 (정보없음 포함) ---
        # 가나다 순 정렬 (정보없음이 포함됨)
        attributes = sorted(p_attr_df['Attribute'].unique())
        for attr in attributes:
            group = p_attr_df[p_attr_df['Attribute'] == attr]
            user_count = group['리뷰ID'].nunique()
            avg_score = round(group['감성점수'].mean(), 2)

            # 소분류(Sub-category) 추출 및 집계
            subcat_df = group.copy()
            subcat_df['Subcat'] = subcat_df['소분류'].astype(str).str.split(", ")
            subcat_df = subcat_df.explode('Subcat')
            subcat_df = subcat_df[subcat_df['Subcat'] != 'nan']

            pos_subs = subcat_df[subcat_df['긍부정'] == '긍정']['Subcat'].value_counts().head(5)
            pos_str = "\n".join([f"{k} ({v})" for k, v in pos_subs.items()]) if not pos_subs.empty else "-"

            neg_subs = subcat_df[subcat_df['긍부정'] == '부정']['Subcat'].value_counts().head(5)
            neg_str = "\n".join([f"{k} ({v})" for k, v in neg_subs.items()]) if not neg_subs.empty else "-"

            final_report_rows.append({
                "제품명": product,
                "상세 항목 및 요약": attr,
                "사용자 수 / 정보밀도": f"{user_count}명",
                "긍정 소분류 (Top 5)": pos_str,
                "부정 소분류 (Top 5)": neg_str,
                "종합 만족도": "매우만족" if avg_score >= 1.5 else "만족" if avg_score >= 0.5 else "보통",
                "평균 점수": avg_score
            })

        # --- PART B: 제품 총평 랩업 (Wrap-up) ---
        # 속성별 유니크 유저 수 집계
        demo_counts = p_attr_df.groupby('Attribute')['리뷰ID'].nunique().sort_values(ascending=False).to_dict()
        demo_str = ", ".join([f"{k} {v}명" for k, v in demo_counts.items()])

        # 제품 전체에서 가장 많이 언급된 특징과 감성
        overall_subcat_df = p_df.copy()
        overall_subcat_df['Subcat'] = overall_subcat_df['소분류'].astype(str).str.split(", ")
        overall_subcat_df = overall_subcat_df.explode('Subcat')
        overall_subcat_df = overall_subcat_df[overall_subcat_df['Subcat'] != 'nan']

        if not overall_subcat_df.empty:
            top_sub = overall_subcat_df['Subcat'].value_counts().idxmax()
            top_sub_score = overall_subcat_df[overall_subcat_df['Subcat'] == top_sub]['감성점수'].mean()
            top_sub_sent = "긍정" if top_sub_score > 0 else "부정"
        else:
            top_sub = "정보없음"
            top_sub_sent = "보통"

        wrap_up_text = f"[{product} 랩업] 참여 유저: ({demo_str}). 이 제품에서 가장 많이 언급된 특징은 '{top_sub}'(으)로, 이에 대해 주로 {top_sub_sent}적인 반응을 보였습니다."

        final_report_rows.append({
            "제품명": product,
            "상세 항목 및 요약": wrap_up_text,
            "사용자 수 / 정보밀도": f"총 {p_df['리뷰ID'].nunique()}명",
            "긍정 소분류 (Top 5)": "★★ 제품 전체 요약 ★★",
            "부정 소분류 (Top 5)": "★★ 제품 전체 요약 ★★",
            "종합 만족도": "-",
            "평균 점수": round(p_df['감성점수'].mean(), 2)
        })

        # --- PART C: 정보 밀도 높은 대표 리뷰 TOP 3 ---
        # 중복 제거 후 정보밀도 순으로 정렬
        top_reviews = p_df.drop_duplicates(subset='리뷰ID').sort_values(by='정보밀도(대분류수)', ascending=False).head(3)

        for i, (_, r_row) in enumerate(top_reviews.iterrows()):
            final_report_rows.append({
                "제품명": product,
                "상세 항목 및 요약": f"[대표리뷰 {i+1}] {r_row['clean_text']}",
                "사용자 수 / 정보밀도": f"밀도: {r_row['정보밀도(대분류수)']}",
                # [수정] 작성자 정보 자리에 고유 리뷰ID를 표시합니다.
                "긍정 소분류 (Top 5)": f"리뷰ID: {r_row['리뷰ID']}",
                "부정 소분류 (Top 5)": f"태그: {r_row['태그']}",
                "종합 만족도": f"별점: {r_row['별점']}",
                "평균 점수": r_row['리뷰_평균점수']
            })

    # 최종 결과 저장 (UTF-8-SIG 인코딩으로 엑셀 한글 깨짐 방지)
    final_df = pd.DataFrame(final_report_rows)
    final_df.to_csv("ampoule_analysis_summary_report.csv", index=False, encoding='utf-8-sig')
    print("✔ 소비자 인사이트 요약 리포트 생성 완료: ampoule_analysis_summary_report.csv")

✔ 소비자 인사이트 요약 리포트 생성 완료: ampoule_analysis_summary_report.csv


### 기타(폼, 선크림, 오일)

In [20]:
import pandas as pd
import os
import re

# ==========================================
# 1. 제품별 식별 코드 매핑 (기타 라인업)
# ==========================================
product_code_map = {
    "reviews_cleaned_11.아누아 어성초 피지쏙 모공 폼 150ml.csv": "FC",
    "reviews_cleaned_10.[파데프리 선크림] 아누아 매트벗글로우 커버 베이지 50ml 2입.csv": "SC",
    "reviews_cleaned_6. 아누아 어성초 포어 컨트롤 클렌징오일 200ml 2입.csv": "CO"
}

# ==========================================
# 2. 고도화된 카테고리 구조 (Taxonomy 2.2)
# ==========================================
category_structure = {
    "피부 고민 해결": {
        "트러블/모공": ["여드름", "좁쌀", "트러블", "뾰루지", "모공", "요철", "블랙헤드", "화이트헤드", "흉터", "흔적", "화농성", "올라옴"],
        "진정/장벽": ["진정", "가라앉", "붉은기", "홍조", "재생", "회복", "장벽", "손상", "보호", "예민", "열감", "쿨링", "편안"],
        "미백/광채": ["미백", "밝아", "환해", "광채", "물광", "속광", "톤업", "잡티", "기미", "투명", "칙칙", "결 개선", "생기"]
    },
    "수분/유분 밸런스": {
        "보습/속건조": ["속건조", "속당김", "당김", "건조", "보습", "촉촉", "수분감", "푸석", "겉돌지", "수분충전", "촘촘", "속보습", "수부지"],
        "유분/피지": ["유분", "기름", "개기름", "번들", "산뜻", "피지", "보송", "오일리", "유분기", "답답함", "기름기"]
    },
    "제형/사용감": {
        "흡수/발림성": ["흡수", "스며", "발림", "제형", "질감", "밀착", "부드럽", "실키", "가벼운", "무거운", "점성", "묽은", "쫀쫀"],
        "끈적임/밀림": ["끈적", "밀림", "밀려", "밀착력", "잔여감", "겉돎", "유분감"]
    },
    "피부 자극": {
        "순함/저자극": ["순함", "순해", "순한", "자극없", "안심", "민감", "무자극", "따갑지", "순하게", "부담없는"],
        "자극/부작용": ["자극", "따갑", "따끔", "가려", "화끈", "뒤집", "붉어짐", "화끈거림", "가렵"]
    },
    "구매 가치/편의": {
        "가성비/가격": ["가격", "가성비", "세일", "혜택", "비쌈", "저렴", "대용량", "구성", "증정", "가심비", "착한", "착하다", "혜자", "이득", "사악"],
        "추천/재구매": ["추천", "강추", "인생템", "정착", "재구매", "지인", "비추", "선물", "만족", "재구입", "공병", "애정템"],
        "상황/편의": ["화장", "메이크업", "화잘먹", "펌프", "냄새", "향", "데일리", "팩토", "닦토", "흡토", "분사"]
    }
}

# ==========================================
# 3. [통합] 감성 사전 및 문맥 차단 로직
# ==========================================
SUPER_POS = ["인생템", "강추", "최고", "정착", "성공", "재구매", "무조건", "대박", "완전 추천", "효과 확실", "공병템", "확실", "아주 좋"]
NORMAL_POS = ["좋", "만족", "순해", "인정", "찰떡", "촉촉", "진정", "개선", "효과", "괜찮", "편안", "매끈", "맑", "탄탄", "믿고", "추천", "착한", "착하다", "혜자", "이득", "쫀쫀", "잘먹"]
SUPER_NEG = ["비추", "돈아깝", "거름", "최악", "다신안", "환불", "반품", "쓰레기", "뒤집어짐", "실망", "속았", "사악"]
NORMAL_NEG = ["아쉬", "안맞", "따가", "비싸", "밀려", "자극", "가려", "좁쌀", "번들", "답답", "냄새", "부족", "별로", "그다지", "불편", "심하", "의심", "부담되는", "들뜸", "뜸"]
INTENSIFIERS = ["너무", "진짜", "완전", "엄청", "진심", "정말", "되게", "매우", "솔직히", "겁나", "아주"]

ABSOLUTE_POS_PHRASES = ["화잘먹", "공병", "정착", "인생", "득템", "잘 먹어", "잘 먹음"]
QUOTE_MARKERS = ["광고", "바이럴", "영상", "유튜브", "남들은", "후기", "기대", "없어졌다고"]
CONTRAST_MARKERS = ["했는데", "했으나", "그치만", "하지만"]

NEGATIVE_REVERSAL = ["없", "않", "적", "사라", "진정", "개선", "나아", "제로", "줄어", "잡혔", "안 나", "안 뜨", "들어갔"]
NEG_TARGETS = ["자극", "끈적", "트러블", "밀림", "따가", "붉은기", "번들", "건조", "당김", "뒤집", "들뜸", "뜸", "유분"]

def smart_split(text):
    text = str(text)
    initial_splits = re.split(r'[.!\?\n]+', text)
    refined = []
    for s in initial_splits:
        s = s.strip()
        if len(s) > 35:
            sub = re.split(r'(?<=[고서데며])\s+', s)
            refined.extend(sub)
        else:
            refined.append(s)
    return [s.strip() for s in refined if len(s.strip()) > 2]

def calculate_refined_sentiment(sentence, matched_kws):
    score = 0
    is_quote = any(q in sentence for q in QUOTE_MARKERS)
    has_contrast = any(c in sentence for c in CONTRAST_MARKERS)

    for wp in SUPER_POS: score += 3 if wp in sentence else 0
    for wp in NORMAL_POS: score += 1 if wp in sentence else 0
    for wn in SUPER_NEG: score -= 3 if wn in sentence else 0
    for wn in NORMAL_NEG: score -= 1 if wn in sentence else 0

    if any(ap in sentence for ap in ABSOLUTE_POS_PHRASES):
        score += 2

    if any(i in sentence for i in INTENSIFIERS):
        if score > 0: score += 1
        elif score < 0: score -= 1

    for target in NEG_TARGETS:
        if target in sentence:
            after_target = sentence.split(target)[-1]
            if not is_quote and any(rev in after_target for rev in NEGATIVE_REVERSAL):
                score = abs(score) + 2
            if any(neg in after_target for neg in ["올라", "나왔", "생겼", "심해", "안 좋아"]):
                score -= 3

    if is_quote and has_contrast:
        if any(n in sentence for n in NORMAL_NEG + SUPER_NEG + ["올라", "안 좋아", "나왔"]):
            score = -4

    if "가격" in sentence or "가성비" in sentence:
        if any(p in sentence for p in ["착한", "착하다", "혜자", "부담없", "저렴", "이득"]):
            score = max(score, 2)
        if any(n in sentence for n in ["사악", "비싸", "부담되"]):
            score = min(score, -2)

    score = int(max(-5, min(5, score)))
    label = "긍정" if score > 0 else "부정" if score < 0 else "보통"
    return label, score

# ==========================================
# 4. 분석 실행 및 리포트 저장
# ==========================================
final_rows = []

for f, p_code in product_code_map.items():
    if not os.path.exists(f): continue
    df = pd.read_csv(f)
    p_name = f.split('.')[-2].strip() if '.' in f else f
    df['날짜'] = df['날짜'].astype(str)
    df = df.sort_values('날짜')
    date_id_counts = {}

    for _, row in df.iterrows():
        date_raw = row['날짜']
        date_clean = date_raw.replace('.', '')[2:8]
        date_id_counts[date_clean] = date_id_counts.get(date_clean, 0) + 1

        review_id = f"{p_code}_{date_clean}_{date_id_counts[date_clean]:03d}"
        raw_text = str(row.get('clean_text', row.get('리뷰내용', '')))
        sentences = smart_split(raw_text)

        review_sentences_data = []
        review_all_kws, review_major_cats = set(), set()

        for sent in sentences:
            sent_major, sent_minor, sent_kws = set(), set(), set()
            for major, submap in category_structure.items():
                for minor, kws in submap.items():
                    matched = [kw for kw in kws if kw in sent]
                    if matched:
                        sent_major.add(major); sent_minor.add(minor); sent_kws.update(matched)
                        review_major_cats.add(major); review_all_kws.update(matched)

            if sent_kws:
                label, score = calculate_refined_sentiment(sent, sent_kws)
                review_sentences_data.append({"문장": sent, "라벨": label, "점수": score, "대": sent_major, "소": sent_minor, "키": sent_kws})

        avg_score = round(sum(d["점수"] for d in review_sentences_data) / len(review_sentences_data), 1) if review_sentences_data else 0.0

        for res in review_sentences_data:
            final_rows.append({
                "리뷰ID": review_id,
                "제품군": "기타",
                "제품명": p_name,
                "날짜": date_raw,
                "가격": row.get('price', row.get('가격', '-')),
                "스킨타입": row.get('스킨타입', '-'),
                "태그": row.get('태그', '-'),
                "대분류": ", ".join(sorted(res["대"])),
                "소분류": ", ".join(sorted(res["소"])),
                "정보밀도(대분류수)": len(review_major_cats),
                "별점": row.get('별점', '-'),
                "긍부정": res["라벨"],
                "감성점수": res["점수"],
                "매칭키워드": ", ".join(sorted(res["키"])),
                "추출문장": res["문장"],
                "clean_text": raw_text,
                "리뷰_평균점수": avg_score,
                "전체_매칭키워드": ", ".join(sorted(review_all_kws))
            })

output_file = "others_analysis_report.csv"
pd.DataFrame(final_rows).to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"✔ 분석 완료: {output_file}")

✔ 분석 완료: others_analysis_report.csv


In [21]:
import pandas as pd
import numpy as np
import os

# 1. 원본 데이터 로드 (기존 문장 단위 분석 결과 파일)
base_file = "others_analysis_report.csv"

if not os.path.exists(base_file):
    print(f"⚠ {base_file} 파일이 없습니다. 분석을 먼저 실행해주세요.")
else:
    df = pd.read_csv(base_file)

    # 2. 속성(스킨타입 + 태그) 통합 및 분리 함수 (정보없음 포함 로직)
    def get_attribute_df(dataframe):
        temp_df = dataframe.copy()

        def combine_attrs(row):
            s = str(row.get('스킨타입', '-')).strip()
            t = str(row.get('태그', '-')).strip()

            # 비어있거나 의미 없는 값 체크
            def is_empty(val):
                return val in ['-', 'nan', 'None', '', 'nan ']

            attrs = []
            if not is_empty(s):
                attrs.extend([item.strip() for item in s.split(",")])
            if not is_empty(t):
                attrs.extend([item.strip() for item in t.split(",")])

            # 둘 다 정보가 없는 경우에만 '정보없음' 할당
            if not attrs:
                return ["정보없음"]
            # 중복 제거 (스킨타입과 태그에 중복된 단어가 있을 경우 대비)
            return list(set(attrs))

        temp_df['Attribute'] = temp_df.apply(combine_attrs, axis=1)
        # 리스트로 묶인 속성들을 행별로 분리(explode)
        temp_df = temp_df.explode('Attribute')
        return temp_df

    attr_df = get_attribute_df(df)
    final_report_rows = []
    products = df['제품명'].unique()

    for product in products:
        p_df = df[df['제품명'] == product]
        p_attr_df = attr_df[attr_df['제품명'] == product]

        # --- PART A: 속성별 수치 요약 (정보없음 포함) ---
        # 가나다 순 정렬 (정보없음이 포함됨)
        attributes = sorted(p_attr_df['Attribute'].unique())
        for attr in attributes:
            group = p_attr_df[p_attr_df['Attribute'] == attr]
            user_count = group['리뷰ID'].nunique()
            avg_score = round(group['감성점수'].mean(), 2)

            # 소분류(Sub-category) 추출 및 집계
            subcat_df = group.copy()
            subcat_df['Subcat'] = subcat_df['소분류'].astype(str).str.split(", ")
            subcat_df = subcat_df.explode('Subcat')
            subcat_df = subcat_df[subcat_df['Subcat'] != 'nan']

            pos_subs = subcat_df[subcat_df['긍부정'] == '긍정']['Subcat'].value_counts().head(5)
            pos_str = "\n".join([f"{k} ({v})" for k, v in pos_subs.items()]) if not pos_subs.empty else "-"

            neg_subs = subcat_df[subcat_df['긍부정'] == '부정']['Subcat'].value_counts().head(5)
            neg_str = "\n".join([f"{k} ({v})" for k, v in neg_subs.items()]) if not neg_subs.empty else "-"

            final_report_rows.append({
                "제품명": product,
                "상세 항목 및 요약": attr,
                "사용자 수 / 정보밀도": f"{user_count}명",
                "긍정 소분류 (Top 5)": pos_str,
                "부정 소분류 (Top 5)": neg_str,
                "종합 만족도": "매우만족" if avg_score >= 1.5 else "만족" if avg_score >= 0.5 else "보통",
                "평균 점수": avg_score
            })

        # --- PART B: 제품 총평 랩업 (Wrap-up) ---
        # 속성별 유니크 유저 수 집계
        demo_counts = p_attr_df.groupby('Attribute')['리뷰ID'].nunique().sort_values(ascending=False).to_dict()
        demo_str = ", ".join([f"{k} {v}명" for k, v in demo_counts.items()])

        # 제품 전체에서 가장 많이 언급된 특징과 감성
        overall_subcat_df = p_df.copy()
        overall_subcat_df['Subcat'] = overall_subcat_df['소분류'].astype(str).str.split(", ")
        overall_subcat_df = overall_subcat_df.explode('Subcat')
        overall_subcat_df = overall_subcat_df[overall_subcat_df['Subcat'] != 'nan']

        if not overall_subcat_df.empty:
            top_sub = overall_subcat_df['Subcat'].value_counts().idxmax()
            top_sub_score = overall_subcat_df[overall_subcat_df['Subcat'] == top_sub]['감성점수'].mean()
            top_sub_sent = "긍정" if top_sub_score > 0 else "부정"
        else:
            top_sub = "정보없음"
            top_sub_sent = "보통"

        wrap_up_text = f"[{product} 랩업] 참여 유저: ({demo_str}). 이 제품에서 가장 많이 언급된 특징은 '{top_sub}'(으)로, 이에 대해 주로 {top_sub_sent}적인 반응을 보였습니다."

        final_report_rows.append({
            "제품명": product,
            "상세 항목 및 요약": wrap_up_text,
            "사용자 수 / 정보밀도": f"총 {p_df['리뷰ID'].nunique()}명",
            "긍정 소분류 (Top 5)": "★★ 제품 전체 요약 ★★",
            "부정 소분류 (Top 5)": "★★ 제품 전체 요약 ★★",
            "종합 만족도": "-",
            "평균 점수": round(p_df['감성점수'].mean(), 2)
        })

        # --- PART C: 정보 밀도 높은 대표 리뷰 TOP 3 ---
        # 중복 제거 후 정보밀도 순으로 정렬
        top_reviews = p_df.drop_duplicates(subset='리뷰ID').sort_values(by='정보밀도(대분류수)', ascending=False).head(3)

        for i, (_, r_row) in enumerate(top_reviews.iterrows()):
            final_report_rows.append({
                "제품명": product,
                "상세 항목 및 요약": f"[대표리뷰 {i+1}] {r_row['clean_text']}",
                "사용자 수 / 정보밀도": f"밀도: {r_row['정보밀도(대분류수)']}",
                # [수정] 작성자 정보 자리에 고유 리뷰ID를 표시합니다.
                "긍정 소분류 (Top 5)": f"리뷰ID: {r_row['리뷰ID']}",
                "부정 소분류 (Top 5)": f"태그: {r_row['태그']}",
                "종합 만족도": f"별점: {r_row['별점']}",
                "평균 점수": r_row['리뷰_평균점수']
            })

    # 최종 결과 저장 (UTF-8-SIG 인코딩으로 엑셀 한글 깨짐 방지)
    final_df = pd.DataFrame(final_report_rows)
    final_df.to_csv("others_analysis_summary_report.csv", index=False, encoding='utf-8-sig')
    print("✔ 소비자 인사이트 요약 리포트 생성 완료: others_analysis_summary_report.csv")

✔ 소비자 인사이트 요약 리포트 생성 완료: others_analysis_summary_report.csv


###